# Representation augmentation — fixing the resource-notation brittleness

E5 showed the model transfers to real tool-call *vocabulary* but is brittle
to resource *notation*: the released model scores 90.8% on tau2 in the
trained `family:namespace/leaf` notation but ~55% on the same data in an
all-colon notation. Here we retrain CE with **notation augmentation** (each
training trace re-notated under a random delimiter scheme; labels
re-verified, provably unchanged) and test whether the model then handles
both notations — i.e. whether the brittleness is fixable, not fundamental.

Plain `!` commands, `python -u` streams live. Runtime → T4/L4 → Run all
(~1.5–2 h for 3 seeds). Download `augment_results.zip` at the end.

In [ ]:
import os
os.chdir('/content')
if not os.path.isdir('verifier-as-reward'):
    !git clone https://github.com/esmaeil-abedi-dev/verifier-as-reward.git
os.chdir('/content/verifier-as-reward')
!git pull --ff-only
import torch; assert torch.cuda.is_available()
!PYTHONPATH=. python test_authority_verifier.py
!PYTHONPATH=. python make_expanded_train.py --train-seed 101 --train-traces-per-class 150 --val-seed 202 --val-traces-per-class 25

## Step 1 — build the notation-augmented training corpus (+ tests)

In [ ]:
!PYTHONPATH=. python -u augment_representation.py --in expanded_train.jsonl --out augmented_train.jsonl --seed 33
!PYTHONPATH=. python test_augment_representation.py

## Step 2 — train CE on the augmented corpus (3 seeds; set `7` only for a quick look)

In [ ]:
!for s in 7 8 9; do echo "=== augment seed $s ==="; PYTHONPATH=. python -u train_verifier_reward.py --model Qwen/Qwen2.5-0.5B --ce-loss --balance-reward --prompt-style nl --steps 500 --batch-size 16 --lr 2e-5 --train-file augmented_train.jsonl --test-file expanded_val.jsonl --eval-every 50 --eval-max-actions 120 --seed $s --log-file training_log_augment_seed$s.jsonl --save-dir ckpt_augment_seed$s || break; done

## Step 3 — build the real-trace test in BOTH notations (slash = trained, colon = naive)

In [ ]:
!PYTHONPATH=. python map_tau_to_chain.py --seed 5   # writes real_trace_*.jsonl (slash)
from trace_benchmark import load_traces, write_jsonl
from augment_representation import renotate_trace
slash = load_traces('real_trace_all.jsonl')
write_jsonl([renotate_trace(t, 'allcolon') for t in slash], 'real_trace_all_colon.jsonl')
print('slash:', slash[0]['actions'][0]['resource'],
      '| colon:', renotate_trace(slash[0], 'allcolon')['actions'][0]['resource'])

## Step 4 — before/after: released model vs augmented model, both notations

In [ ]:
import json, os
json.dump({'backends': {}}, open('results_augment.json', 'w'))
# BEFORE: the released (non-augmented) model on both notations
!PYTHONPATH=. python train_verifier_reward.py --eval-checkpoint esmaeil-abedi-dev/verifier-ce-qwen2.5-0.5b --test-file real_trace_all.jsonl --merge-results results_augment.json
!PYTHONPATH=. python train_verifier_reward.py --eval-checkpoint esmaeil-abedi-dev/verifier-ce-qwen2.5-0.5b --test-file real_trace_all_colon.jsonl --merge-results results_augment.json
# AFTER: each augmented checkpoint on committed test + both real notations
!for s in 7 8 9; do ck=ckpt_augment_seed$s; if [ -d $ck ]; then for tf in benchmark_test.jsonl real_trace_all.jsonl real_trace_all_colon.jsonl; do echo "=== $ck on $tf ==="; PYTHONPATH=. python train_verifier_reward.py --eval-checkpoint $ck --test-file $tf --merge-results results_augment.json; done; fi; done

In [ ]:
import json
from stats import wilson_ci
d = json.load(open('results_augment.json'))
print(f"{'checkpoint / test':60} {'n':>4} {'acc% [CI]':>20} {'f-auth%':>8}")
for name in sorted(d['backends']):
    m = d['backends'][name]['metrics']; n = m['n_actions']; c = wilson_ci(round(m['accuracy']*n), n)
    fa = m['false_authorize_rate']
    print(f"{name.replace('local:','')[:60]:60} {n:4d} {100*m['accuracy']:5.1f} [{100*c[0]:.1f},{100*c[1]:.1f}] {'n/a' if fa is None else format(100*fa,'6.1f')}")
print('\nKEY: released model should be ~90% slash / ~55% colon (brittle);')
print('augmented model should be high on BOTH (robust) — that is the result.')

In [ ]:
!zip -j augment_results.zip training_log_augment_*.jsonl results_augment.json
from google.colab import files
files.download('augment_results.zip')